# Custom Chatbot Project

I chose the **2023 Fashion Trends** dataset (`data/2023_fashion_trends.csv`), which contains 93 rows of detailed descriptions of fashion trends for 2023 along with their sources. This dataset is well-suited for a RAG chatbot for several reasons: (1) it exceeds the 20-row minimum requirement, providing a rich corpus of domain-specific text; (2) the content covers specific, time-bound fashion trends (e.g., Spring 2023 denim styles, color palettes) that a base language model is unlikely to know in detail, making the contrast between basic and custom query results very clear; and (3) the text entries are descriptive and varied enough to produce meaningful embeddings for similarity-based retrieval.

## Data Wrangling

TODO: In the cells below, load your chosen dataset into a `pandas` dataframe with a column named `"text"`. This column should contain all of your text data, separated into at least 20 rows.

In [1]:
import pandas as pd
df = pd.read_csv("data/2023_fashion_trends.csv")
df.head()

,URL,Trends,Source
0,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Red. Glossy red hues took ...,7 Fashion Trends That Will Take Over 2023 — Sh...
1,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Cargo Pants. Utilitarian w...,7 Fashion Trends That Will Take Over 2023 — Sh...
2,https://www.refinery29.com/en-us/fashion-trend...,"2023 Fashion Trend: Sheer Clothing. ""Bare it a...",7 Fashion Trends That Will Take Over 2023 — Sh...
3,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Denim Reimagined. From dou...,7 Fashion Trends That Will Take Over 2023 — Sh...
4,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Shine For The Daytime. The...,7 Fashion Trends That Will Take Over 2023 — Sh...


In [2]:
df["text"] = df["Trends"] + " (Source: " + df["Source"] + ")"
df = df[["text"]]
df.head()

,text
0,2023 Fashion Trend: Red. Glossy red hues took ...
1,2023 Fashion Trend: Cargo Pants. Utilitarian w...
2,"2023 Fashion Trend: Sheer Clothing. ""Bare it a..."
3,2023 Fashion Trend: Denim Reimagined. From dou...
4,2023 Fashion Trend: Shine For The Daytime. The...


In [3]:
print(f"Number of rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

Number of rows: 82
Columns: ['text']


## Custom Query Completion

TODO: In the cells below, compose a custom query using your chosen dataset and retrieve results from an OpenAI `Completion` model. You may copy and paste any useful code from the course materials.

In [ ]:
from openai import OpenAI
import numpy as np
import tiktoken

client = OpenAI(
    base_url="https://openai.vocareum.com/v1",
    api_key="YOUR API KEY"
)
EMBEDDING_MODEL = "text-embedding-ada-002"

In [5]:
def get_embedding(text, model=EMBEDDING_MODEL):
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [6]:
df["embedding"] = df["text"].apply(lambda x: get_embedding(x))
df.head()

,text,embedding
0,2023 Fashion Trend: Red. Glossy red hues took ...,"[-0.017936429008841515, -0.02061583660542965, ..."
1,2023 Fashion Trend: Cargo Pants. Utilitarian w...,"[-0.0008542184950783849, -0.030208196491003036..."
2,"2023 Fashion Trend: Sheer Clothing. ""Bare it a...","[-0.009967019781470299, -0.019867325201630592,..."
3,2023 Fashion Trend: Denim Reimagined. From dou...,"[-0.012048566713929176, -0.007959355600178242,..."
4,2023 Fashion Trend: Shine For The Daytime. The...,"[-0.003422758774831891, 0.0017147186445072293,..."


In [8]:
def search_text(df, query, n=3):
    query_embedding = get_embedding(query)
    df["similarity"] = df["embedding"].apply(lambda x: cosine_similarity(x, query_embedding))
    return df.sort_values("similarity", ascending=False).head(n)

def create_prompt(question, df, max_token_count=1800):
    results = search_text(df, question)
    tokenizer = tiktoken.get_encoding("cl100k_base")
    context = ""
    for _, row in results.iterrows():
        text = row["text"]
        if len(tokenizer.encode(context + text)) < max_token_count:
            context += text + "\n\n"
        else:
            break
    return f"""Use the below articles on 2023 fashion trends to answer the question. If the answer cannot be found, write "I could not find an answer."

Context:
\"\"\"
{context}\"\"\"

Question: {question}"""

In [9]:
def answer_question(question, df, model="gpt-3.5-turbo"):
    prompt = create_prompt(question, df)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0
    )
    return response.choices[0].message.content

## Custom Performance Demonstration

TODO: In the cells below, demonstrate the performance of your custom query using at least 2 questions. For each question, show the answer from a basic `Completion` model query as well as the answer from your custom query.

### Question 1

In [10]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "What are the key denim trends for spring 2023?"}],
    max_tokens=500,
    temperature=0
)
print("BASIC ANSWER:\n")
print(response.choices[0].message.content)

BASIC ANSWER:

1. Wide-leg jeans: Loose-fitting, wide-leg jeans are set to be a key trend for spring 2023, offering a relaxed and comfortable alternative to skinny jeans.

2. Patchwork denim: Patchwork denim, featuring different washes and textures of denim pieced together, will be a popular trend for adding a unique and eclectic touch to your denim wardrobe.

3. High-waisted jeans: High-waisted jeans continue to be a staple trend, offering a flattering and versatile option that can be dressed up or down for any occasion.

4. Distressed denim: Distressed denim, featuring frayed hems, ripped knees, and worn-in details, will continue to be a popular trend for adding a cool and edgy vibe to your look.

5. Denim sets: Coordinating denim sets, featuring matching jackets and jeans or skirts, will be a key trend for spring 2023, offering a chic and effortless way to wear denim from head to toe.

6. Vintage-inspired denim: Vintage-inspired denim, featuring retro silhouettes and washes from pas

In [11]:
print("CUSTOM ANSWER:\n")
print(answer_question("What are the key denim trends for spring 2023?", df))

CUSTOM ANSWER:

The key denim trends for spring 2023 include double-waisted jeans, carpenter jeans, strapless dresses, shirting, undergarments, shoes like thigh-high-boot-jean hybrids, timeless cuts, and silhouettes. Additionally, detailed denim such as not-so-basic denim pieces like maxiskirts are also trending. Loose-fit denim with wide-legs, puddle hemlines, and slouchy-fit trousers are popular for spring 2023.


### Question 2

In [12]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "What color trends are popular in spring 2023 fashion?"}],
    max_tokens=500,
    temperature=0
)
print("BASIC ANSWER:\n")
print(response.choices[0].message.content)

BASIC ANSWER:

Some popular color trends in spring 2023 fashion include:

1. Pastel shades: Soft and delicate pastel colors like lavender, mint green, baby blue, and blush pink are popular for spring.

2. Bright and bold hues: Vibrant colors like electric blue, neon green, hot pink, and sunny yellow are also trending for spring 2023.

3. Earthy tones: Natural and earthy colors such as terracotta, olive green, rust, and mustard yellow are popular for a more grounded and organic look.

4. Metallics: Shiny metallic colors like silver, gold, and bronze are making a statement in spring 2023 fashion, adding a touch of glamour to outfits.

5. Monochromatic looks: Wearing head-to-toe outfits in a single color, known as monochromatic dressing, is a popular trend for spring 2023. This can be done with any color, from classic black or white to more bold shades.

Overall, the color trends for spring 2023 are a mix of soft pastels, bold brights, earthy tones, metallics, and monochromatic looks, off

In [13]:
print("CUSTOM ANSWER:\n")
print(answer_question("What color trends are popular in spring 2023 fashion?", df))

CUSTOM ANSWER:

The color trends popular in spring 2023 fashion are cobalt blue and red.
